# BioVerify — Google Colab Runner
**Evidence-Aware Biomedical Answer Verification with Uncertainty Detection for Patient-Safe Healthcare NLP**

---

## ⚠️ Before You Start — Read This

This notebook is designed for **Google Colab free tier (T4 GPU)**.  
**Do NOT run all cells at once.** Each section is independent — run only what you need.

### Recommended workflow
| Step | Section | Notes |
|------|---------|-------|
| 1 | Upload dataset to Google Drive | Do this once, outside Colab |
| 2 | **Mount Google Drive** | Always run first |
| 3 | **GPU Check** | Confirm T4 is assigned |
| 4 | **Clone Repo** | Run once per session |
| 5 | **Install Dependencies** | Run once per session |
| 6 | **Dataset Setup** | Verify paths before any training |
| 7 | **Train: TF-IDF + LR** | Fast baseline (~2 min) |
| 8 | **Train: DistilBERT** | ~15 min on T4 |
| 9 | **Train: BioBERT** | ~25 min on T4 — restart runtime first |
| 10 | **Train: PubMedBERT** (proposed) | ~25 min on T4 — restart runtime first |
| 11 | **Evaluate** | Run after all models trained |
| 12 | **Uncertainty Analysis** | Per-model, run after evaluation |
| 13 | **Safety Analysis** | Per-model, run after uncertainty |
| 14 | **Ablation Study** | Run after PubMedBERT is trained |
| 15 | **Cross-Model Comparison** | Run last |
| 16 | **Inference** | Run on any new inputs |
| 17 | **Visualize Results** | Display saved figures |

### Google Drive folder layout expected
```
MyDrive/BioVerify/
    datasets/
        ori_pqal.json          ← upload this manually
    checkpoints/
    outputs/
    logs/
    results/
```

## 🔧 Section 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Edit these if your Drive folder is named differently ────────────
PROJECT_NAME    = 'BioVerify'
DRIVE_ROOT      = f'/content/drive/MyDrive/{PROJECT_NAME}'

DATA_ROOT       = f'{DRIVE_ROOT}/datasets'
CHECKPOINT_ROOT = f'{DRIVE_ROOT}/checkpoints'
OUTPUT_ROOT     = f'{DRIVE_ROOT}/outputs'
LOG_ROOT        = f'{DRIVE_ROOT}/logs'
RESULTS_ROOT    = f'{DRIVE_ROOT}/results'
# ────────────────────────────────────────────────────────────────────

# Create all folders if they do not exist
for folder in [
    DATA_ROOT,
    f'{CHECKPOINT_ROOT}/tfidf_lr',
    f'{CHECKPOINT_ROOT}/distilbert',
    f'{CHECKPOINT_ROOT}/biobert',
    f'{CHECKPOINT_ROOT}/pubmedbert',
    f'{OUTPUT_ROOT}/main_model',
    f'{OUTPUT_ROOT}/baselines',
    f'{OUTPUT_ROOT}/ablations',
    f'{OUTPUT_ROOT}/inference',
    f'{OUTPUT_ROOT}/visualizations',
    f'{OUTPUT_ROOT}/figures',
    f'{OUTPUT_ROOT}/tables',
    LOG_ROOT,
    RESULTS_ROOT,
]:
    os.makedirs(folder, exist_ok=True)

print('✓ Google Drive mounted')
print(f'  DRIVE_ROOT      : {DRIVE_ROOT}')
print(f'  DATA_ROOT       : {DATA_ROOT}')
print(f'  CHECKPOINT_ROOT : {CHECKPOINT_ROOT}')
print(f'  OUTPUT_ROOT     : {OUTPUT_ROOT}')
print(f'  LOG_ROOT        : {LOG_ROOT}')

## 🖥️ Section 2 — GPU Check

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'  CUDA : {torch.version.cuda}')
    DEVICE = 'cuda'
else:
    print('⚠️  No GPU detected — training transformers will be very slow.')
    print('   Go to Runtime → Change runtime type → GPU (T4)')
    DEVICE = 'cpu'

print(f'  PyTorch : {torch.__version__}')
print(f'  Device  : {DEVICE}')

## 📦 Section 3 — Clone GitHub Repository

In [ ]:
import os

REPO_URL  = 'https://github.com/Username/ProjectName.git'  # ← edit if needed
REPO_NAME = 'BioVerify'
REPO_DIR  = f'/content/{REPO_NAME}'

if os.path.exists(REPO_DIR):
    print('Repository already cloned — pulling latest changes...')
    !git -C {REPO_DIR} pull
else:
    print('Cloning repository...')
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'\n✓ Working directory: {os.getcwd()}')
!ls -la

## 📚 Section 4 — Install Dependencies

In [ ]:
import os

# Colab already has torch/torchvision — we install the rest
# The pinned versions in requirements.txt may conflict with Colab's torch;
# we install without strict version pins for compatibility.
print('Installing BioVerify dependencies...')
!pip install -q \
    transformers \
    tokenizers \
    datasets \
    accelerate \
    scikit-learn \
    imbalanced-learn \
    numpy \
    pandas \
    scipy \
    matplotlib \
    seaborn \
    tqdm \
    lime \
    shap

# Verify key imports
import transformers, sklearn, pandas, numpy
print(f'\n✓ transformers : {transformers.__version__}')
print(f'✓ scikit-learn  : {sklearn.__version__}')
print(f'✓ pandas        : {pandas.__version__}')
print(f'✓ numpy         : {numpy.__version__}')
print('\n✓ All dependencies installed.')

## 📂 Section 5 — Dataset Setup & Path Verification

In [ ]:
import os, json, shutil

# Expected raw data file
DRIVE_DATA_FILE   = f'{DATA_ROOT}/ori_pqal.json'
REPO_DATA_DIR     = 'data/pubmedqa'
REPO_DATA_FILE    = 'data/pubmedqa/ori_pqal.json'
PROCESSED_DIR     = 'data/pubmedqa/processed'

# ── Check the file exists on Drive ─────────────────────────────────
if not os.path.exists(DRIVE_DATA_FILE):
    raise FileNotFoundError(
        f'\n❌ ori_pqal.json not found at:\n   {DRIVE_DATA_FILE}\n\n'
        'Please upload it to Google Drive first:\n'
        '  1. Download from: https://github.com/pubmedqa/pubmedqa (data/ori_pqal.json)\n'
        f'  2. Upload to MyDrive/BioVerify/datasets/ori_pqal.json'
    )
print(f'✓ Raw data found: {DRIVE_DATA_FILE}')

# ── Copy to repo's expected location ──────────────────────────────
os.makedirs(REPO_DATA_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
if not os.path.exists(REPO_DATA_FILE):
    shutil.copy(DRIVE_DATA_FILE, REPO_DATA_FILE)
    print(f'✓ Copied to repo: {REPO_DATA_FILE}')
else:
    print(f'✓ Already in repo: {REPO_DATA_FILE}')

# ── Quick validation of JSON structure ────────────────────────────
with open(REPO_DATA_FILE) as f:
    d = json.load(f)
k = next(iter(d))
print(f'\nPubMedQA summary:')
print(f'  Total samples : {len(d)}')
print(f'  Sample PMID   : {k}')
print(f'  Keys          : {list(d[k].keys())}')

decisions = [v['final_decision'] for v in d.values()]
for label in ['yes', 'no', 'maybe']:
    print(f'  {label:>5} : {decisions.count(label)}')

# ── Check if reformulated splits already exist ────────────────────
split_files = ['reformulated_train.csv', 'reformulated_val.csv', 'reformulated_test.csv']
existing = [f for f in split_files if os.path.exists(os.path.join(PROCESSED_DIR, f))]
print(f'\nProcessed splits already in repo: {existing}')
if len(existing) == 3:
    import pandas as pd
    for f in split_files:
        df = pd.read_csv(os.path.join(PROCESSED_DIR, f))
        print(f'  {f}: {len(df)} rows | labels: {dict(df["label"].value_counts())}')
else:
    print('  Splits will be generated automatically on first training run.')

## ⚙️ Section 6 — Configuration Variables

In [ ]:
# ── Editable top-level variables for all training cells ────────────
# These override src/config.py defaults where needed.

MODEL_NAME   = 'pubmedbert'   # tfidf_lr | distilbert | biobert | pubmedbert
BATCH_SIZE   = 16             # Default: 16. Reduce to 8 if you get OOM errors.
NUM_EPOCHS   = 10             # Default: 10. Use 2-3 for quick experiments.
MAX_LENGTH   = 512            # Token sequence length — do not change.
LEARNING_RATE = '2e-5'        # AdamW LR — matches STABLE §8 default.
SEED         = 42
DEVICE       = 'cuda'         # Set to 'cpu' if no GPU.

# Colab-safe fast-dev run (set to None for full training)
# MAX_STEPS = 10   ← uncomment to test pipeline in ~2 min instead of full training
MAX_STEPS    = None

print('Configuration:')
print(f'  MODEL_NAME    = {MODEL_NAME}')
print(f'  BATCH_SIZE    = {BATCH_SIZE}')
print(f'  NUM_EPOCHS    = {NUM_EPOCHS}')
print(f'  MAX_LENGTH    = {MAX_LENGTH}')
print(f'  LEARNING_RATE = {LEARNING_RATE}')
print(f'  SEED          = {SEED}')
print(f'  DEVICE        = {DEVICE}')
print(f'  MAX_STEPS     = {MAX_STEPS} (None = full training)')

# Checkpoint paths on Drive (used to save/restore between sessions)
CKPT_DRIVE = {
    'tfidf_lr'  : f'{CHECKPOINT_ROOT}/tfidf_lr/best_model.pkl',
    'distilbert': f'{CHECKPOINT_ROOT}/distilbert/best_model.pth',
    'biobert'   : f'{CHECKPOINT_ROOT}/biobert/best_model.pth',
    'pubmedbert': f'{CHECKPOINT_ROOT}/pubmedbert/best_model.pth',
}
# Repo-internal checkpoint paths (where src/train.py saves)
CKPT_REPO = {
    'tfidf_lr'  : 'experiments/tfidf_lr/checkpoints/best_model.pkl',
    'distilbert': 'experiments/distilbert/checkpoints/best_model.pth',
    'biobert'   : 'experiments/biobert/checkpoints/best_model.pth',
    'pubmedbert': 'experiments/pubmedbert/checkpoints/best_model.pth',
}
print('\nDrive checkpoint paths:')
for m, p in CKPT_DRIVE.items():
    exists = '✓' if os.path.exists(p) else '—'
    print(f'  {exists} {m}: {p}')

## 🔄 Helper: Sync Checkpoints Between Repo and Drive
_Run this after any training to back up checkpoints to Drive,  
or before evaluation to restore from Drive._

In [ ]:
import shutil, os

def save_checkpoint_to_drive(model_name: str) -> None:
    """Copy best_model.{pth,pkl} from repo experiments/ to Google Drive."""
    src = CKPT_REPO[model_name]
    dst = CKPT_DRIVE[model_name]
    if not os.path.exists(src):
        print(f'  ⚠️  No checkpoint found at {src}')
        return
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)
    size_mb = os.path.getsize(dst) / 1e6
    print(f'  ✓ Saved {model_name} checkpoint to Drive ({size_mb:.1f} MB)')


def restore_checkpoint_from_drive(model_name: str) -> bool:
    """Copy best_model from Drive back into repo experiments/."""
    src = CKPT_DRIVE[model_name]
    dst = CKPT_REPO[model_name]
    if not os.path.exists(src):
        print(f'  ⚠️  No Drive checkpoint for {model_name}')
        return False
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)
    print(f'  ✓ Restored {model_name} checkpoint from Drive')
    return True


def save_outputs_to_drive() -> None:
    """Copy the entire outputs/ tree to Drive."""
    src = 'outputs'
    dst = f'{OUTPUT_ROOT}'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'  ✓ outputs/ synced to {dst}')


def restore_outputs_from_drive() -> None:
    """Restore outputs/ from Drive (useful after session restart)."""
    src = f'{OUTPUT_ROOT}'
    dst = 'outputs'
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'  ✓ outputs/ restored from {src}')


print('Helper functions defined:')
print('  save_checkpoint_to_drive(model_name)')
print('  restore_checkpoint_from_drive(model_name)')
print('  save_outputs_to_drive()')
print('  restore_outputs_from_drive()')

## 🧹 Helper: Memory Cleanup
_Run between heavy training sessions to free GPU VRAM._

In [ ]:
import gc, torch

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved  = torch.cuda.memory_reserved() / 1e9
        print(f'✓ CUDA cache cleared — allocated: {allocated:.2f} GB, reserved: {reserved:.2f} GB')
    else:
        print('✓ CPU memory collected.')

# Run cleanup now
cleanup_memory()

---
## 🏋️ Section 7 — Train Baseline: TF-IDF + Logistic Regression
_Fast (~2 min, CPU only). Recommended as a warm-up before transformer training._

In [ ]:
import os, shutil
os.chdir('/content/BioVerify')

print('Training TF-IDF + Logistic Regression...')
!python src/main.py --mode train --model tfidf_lr

# Backup to Drive
save_checkpoint_to_drive('tfidf_lr')

# Copy logs to Drive
src_log = 'experiments/tfidf_lr/logs/train_log.csv'
if os.path.exists(src_log):
    shutil.copy2(src_log, f'{LOG_ROOT}/tfidf_lr_train_log.csv')
    print(f'✓ Log saved to Drive')

print('\n✓ TF-IDF training complete.')

---
## 🏋️ Section 8 — Train Baseline: DistilBERT
_General-domain transformer baseline. ~15 min on T4._  
_⚠️ Restart runtime and re-run Sections 1–6 before this cell if switching from TF-IDF._

In [ ]:
import os, shutil
os.chdir('/content/BioVerify')

# ── Colab-safe parameters ──────────────────────────────────────────
_EPOCHS     = NUM_EPOCHS   # default 10; reduce to 3 for quick test
_MAX_STEPS  = MAX_STEPS    # None = full epoch; set e.g. 10 for smoke test

cmd = f'python src/main.py --mode train --model distilbert --epochs {_EPOCHS}'
if _MAX_STEPS:
    cmd += f' --max_steps {_MAX_STEPS}'

print(f'Running: {cmd}')
!{cmd}

# Backup checkpoint + logs to Drive
save_checkpoint_to_drive('distilbert')

for fname in ['train_log.csv', 'training_config.json']:
    src = f'experiments/distilbert/logs/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, f'{LOG_ROOT}/distilbert_{fname}')

shutil.copy2('experiments/distilbert/results.json',
             f'{RESULTS_ROOT}/distilbert_results.json') if os.path.exists('experiments/distilbert/results.json') else None

cleanup_memory()
print('\n✓ DistilBERT training complete.')

---
## 🏋️ Section 9 — Train Baseline: BioBERT
_Biomedical transformer. ~25 min on T4. 108M parameters._  
_⚠️ Restart runtime and re-run Sections 1–6 before this cell._

In [ ]:
import os, shutil
os.chdir('/content/BioVerify')

# Resume from Drive checkpoint if available
if restore_checkpoint_from_drive('biobert'):
    print('Resuming from existing BioBERT checkpoint — skipping training.')
    print('Delete the Drive checkpoint to retrain from scratch.')
else:
    _EPOCHS    = NUM_EPOCHS
    _MAX_STEPS = MAX_STEPS

    cmd = f'python src/main.py --mode train --model biobert --epochs {_EPOCHS}'
    if _MAX_STEPS:
        cmd += f' --max_steps {_MAX_STEPS}'

    print(f'Running: {cmd}')
    !{cmd}

    save_checkpoint_to_drive('biobert')

    for fname in ['train_log.csv', 'training_config.json']:
        src = f'experiments/biobert/logs/{fname}'
        if os.path.exists(src):
            shutil.copy2(src, f'{LOG_ROOT}/biobert_{fname}')

    shutil.copy2('experiments/biobert/results.json',
                 f'{RESULTS_ROOT}/biobert_results.json') if os.path.exists('experiments/biobert/results.json') else None

cleanup_memory()
print('\n✓ BioBERT training complete.')

---
## 🏋️ Section 10 — Train Proposed Model: PubMedBERT ⭐
_Primary proposed model. ~25 min on T4. 109M parameters._  
_⚠️ Restart runtime and re-run Sections 1–6 before this cell._

In [ ]:
import os, shutil
os.chdir('/content/BioVerify')

# Resume from Drive checkpoint if available
if restore_checkpoint_from_drive('pubmedbert'):
    print('Resuming from existing PubMedBERT checkpoint — skipping training.')
    print('Delete the Drive checkpoint to retrain from scratch.')
else:
    _EPOCHS    = NUM_EPOCHS
    _MAX_STEPS = MAX_STEPS

    cmd = f'python src/main.py --mode train --model pubmedbert --epochs {_EPOCHS}'
    if _MAX_STEPS:
        cmd += f' --max_steps {_MAX_STEPS}'

    print(f'Running: {cmd}')
    !{cmd}

    save_checkpoint_to_drive('pubmedbert')

    for fname in ['train_log.csv', 'training_config.json']:
        src = f'experiments/pubmedbert/logs/{fname}'
        if os.path.exists(src):
            shutil.copy2(src, f'{LOG_ROOT}/pubmedbert_{fname}')

    shutil.copy2('experiments/pubmedbert/results.json',
                 f'{RESULTS_ROOT}/pubmedbert_results.json') if os.path.exists('experiments/pubmedbert/results.json') else None

cleanup_memory()
print('\n✓ PubMedBERT training complete.')

---
## 📊 Section 11 — Evaluation
_Evaluate each model on the test set. Run per model after training._  
_Change `EVAL_MODEL` to the model you want to evaluate._

In [ ]:
import os, shutil
os.chdir('/content/BioVerify')

EVAL_MODEL = 'pubmedbert'   # ← change to: tfidf_lr | distilbert | biobert | pubmedbert

# Restore checkpoint from Drive if not already in repo
if not os.path.exists(CKPT_REPO[EVAL_MODEL]):
    if not restore_checkpoint_from_drive(EVAL_MODEL):
        raise FileNotFoundError(
            f'No checkpoint found for {EVAL_MODEL}.\n'
            f'Train it first (Section 7–10) or upload checkpoint to Drive.'
        )

print(f'Evaluating {EVAL_MODEL} on test set...')
!python src/main.py --mode evaluate --model {EVAL_MODEL}

# Copy results and figures to Drive
results_src = f'experiments/{EVAL_MODEL}/results.json'
if os.path.exists(results_src):
    shutil.copy2(results_src, f'{RESULTS_ROOT}/{EVAL_MODEL}_results.json')
    print(f'✓ Results saved to Drive')

shutil.copytree('outputs/figures', f'{OUTPUT_ROOT}/figures', dirs_exist_ok=True)
shutil.copytree('outputs/tables',  f'{OUTPUT_ROOT}/tables',  dirs_exist_ok=True)
print(f'✓ Figures and tables synced to Drive')

# Show key metrics
import json
if os.path.exists(results_src):
    r = json.load(open(results_src))
    print(f'\n  Accuracy       : {r.get("accuracy", "—"):.4f}')
    print(f'  Macro-F1       : {r.get("macro_f1", "—"):.4f}')
    print(f'  Contra-F1      : {r.get("contradiction_f1", "—"):.4f}')
    pcf = r.get("per_class_f1") or {}
    print(f'  Supported-F1   : {pcf.get("supported", "—"):.4f}')
    print(f'  Uncertain-F1   : {pcf.get("uncertain", "—"):.4f}')

---
## 🎯 Section 12 — Uncertainty Analysis
_Tune confidence threshold τ on val set; apply to test set; compute ECE._

In [ ]:
import os, shutil
os.chdir('/content/BioVerify')

UNC_MODEL = 'pubmedbert'   # ← change to any trained model

if not os.path.exists(CKPT_REPO[UNC_MODEL]):
    if not restore_checkpoint_from_drive(UNC_MODEL):
        raise FileNotFoundError(f'No checkpoint for {UNC_MODEL}. Train first.')

print(f'Running uncertainty analysis for {UNC_MODEL}...')
!python src/main.py --mode uncertainty --model {UNC_MODEL}

# Sync to Drive
shutil.copytree('outputs/figures', f'{OUTPUT_ROOT}/figures', dirs_exist_ok=True)
shutil.copy2('outputs/tables/uncertainty_detection.csv',
             f'{RESULTS_ROOT}/uncertainty_detection.csv')
print('✓ Uncertainty results saved to Drive')

# Print table
import pandas as pd
df = pd.read_csv('outputs/tables/uncertainty_detection.csv')
print('\nUncertainty detection table:')
print(df.to_string(index=False))

---
## 🛡️ Section 13 — Patient-Safety Layer Analysis
_Assign safety flags (safe / unsafe / expert_review) and compute safety metrics._

In [ ]:
import os, shutil
os.chdir('/content/BioVerify')

SAFETY_MODEL = 'pubmedbert'  # ← change to any trained model

if not os.path.exists(CKPT_REPO[SAFETY_MODEL]):
    if not restore_checkpoint_from_drive(SAFETY_MODEL):
        raise FileNotFoundError(f'No checkpoint for {SAFETY_MODEL}. Train first.')

print(f'Running safety analysis for {SAFETY_MODEL}...')
!python src/main.py --mode safety --model {SAFETY_MODEL}

# Sync
shutil.copytree('outputs/figures', f'{OUTPUT_ROOT}/figures', dirs_exist_ok=True)
shutil.copy2('outputs/tables/safety_evaluation.csv',
             f'{RESULTS_ROOT}/safety_evaluation.csv')
print('✓ Safety results saved to Drive')

import pandas as pd
df = pd.read_csv('outputs/tables/safety_evaluation.csv')
print('\nSafety evaluation table:')
print(df[['model','unsafe_catch_rate','false_safe_rate','expert_review_rate']].to_string(index=False))

---
## 🔬 Section 14 — Ablation Study
_Compare four pipeline variants for the best trained model._

| Variant | Description |
|---------|-------------|
| `full_pipeline` | Model + uncertainty override + safety layer |
| `without_uncertainty` | Skip confidence threshold override |
| `without_safety` | Skip safety keyword flagging |
| `without_both` | Raw model predictions only |

_Note: The ablation study runs inside `compare_results.py` automatically  
using the best available trained model checkpoint._

In [ ]:
import os, shutil, pandas as pd
os.chdir('/content/BioVerify')

# Restore the best model checkpoint if needed (prefer pubmedbert → biobert → distilbert)
for candidate in ['pubmedbert', 'biobert', 'distilbert', 'tfidf_lr']:
    if os.path.exists(CKPT_REPO[candidate]):
        best_model = candidate
        break
    if restore_checkpoint_from_drive(candidate):
        best_model = candidate
        break
else:
    raise FileNotFoundError('No trained checkpoints found. Train at least one model first.')

print(f'Running ablation study using: {best_model}')
print('(Ablation is embedded in compare_results.py — running full comparison now)')
!python src/main.py --mode compare

# Save ablation table to Drive
ablation_src = 'outputs/tables/ablation_study.csv'
if os.path.exists(ablation_src):
    shutil.copy2(ablation_src, f'{RESULTS_ROOT}/ablation_study.csv')
    print('\nAblation study results:')
    df = pd.read_csv(ablation_src)
    cols = [c for c in ['variant','accuracy','macro_f1','contradiction_f1',
                        'uncertainty_recall','false_safe_rate'] if c in df.columns]
    print(df[cols].to_string(index=False))

---
## 📋 Section 15 — Cross-Model Comparison Tables
_Loads all available results.json files and generates the full comparison suite._

In [ ]:
import os, shutil, pandas as pd
os.chdir('/content/BioVerify')

# Restore all available checkpoints and results from Drive
for model in ['tfidf_lr', 'distilbert', 'biobert', 'pubmedbert']:
    restore_checkpoint_from_drive(model)
    # Also restore results.json if present
    drive_results = f'{RESULTS_ROOT}/{model}_results.json'
    repo_results  = f'experiments/{model}/results.json'
    if os.path.exists(drive_results) and not os.path.exists(repo_results):
        os.makedirs(f'experiments/{model}', exist_ok=True)
        shutil.copy2(drive_results, repo_results)
        print(f'  Restored results.json for {model}')

print('\nRunning comparison...')
!python src/main.py --mode compare

# Sync all tables to Drive
shutil.copytree('outputs/tables',  f'{OUTPUT_ROOT}/tables',  dirs_exist_ok=True)
shutil.copytree('outputs/figures', f'{OUTPUT_ROOT}/figures', dirs_exist_ok=True)
print('\n✓ All tables and figures synced to Drive')

# Display main results
print('\n── Main Results ──')
main_tbl = 'outputs/tables/main_results.csv'
if os.path.exists(main_tbl):
    df = pd.read_csv(main_tbl)
    cols = [c for c in ['model','accuracy','macro_f1','f1_contradicted','f1_uncertain'] if c in df.columns]
    print(df[cols].to_string(index=False))

print('\n── Contradiction Focus ──')
contra_tbl = 'outputs/tables/contradiction_focus.csv'
if os.path.exists(contra_tbl):
    print(pd.read_csv(contra_tbl).to_string(index=False))

---
## 🔮 Section 16 — Inference on New Inputs
_Run the full BioVerify pipeline on your own biomedical questions._

In [ ]:
import os, json, shutil
os.chdir('/content/BioVerify')

INFER_MODEL = 'pubmedbert'  # ← change to any trained model

# ── Define your input samples here ──────────────────────────────────
samples = [
    {
        "question": "Does aspirin reduce the risk of heart attack?",
        "evidence": "A randomized trial of 22,071 participants showed aspirin reduced first MI risk by 44%. The study found statistically significant reduction in myocardial infarction events in the aspirin group compared to placebo.",
        "candidate_answer": "Yes, aspirin reduces heart attack risk based on trial evidence."
    },
    {
        "question": "Is it safe to take ibuprofen during pregnancy?",
        "evidence": "NSAIDs including ibuprofen may affect fetal renal function and have been associated with premature closure of the ductus arteriosus when used in the third trimester.",
        "candidate_answer": "No. The evidence does not support this conclusion."
    },
    {
        "question": "Does vitamin C supplementation prevent the common cold?",
        "evidence": "Meta-analyses suggest vitamin C may reduce cold duration slightly but evidence for prevention in general populations is inconsistent and inconclusive.",
        "candidate_answer": "The evidence is inconclusive regarding vitamin C for cold prevention."
    }
]
# ─────────────────────────────────────────────────────────────────────

# Save input JSON
input_path = 'data/colab_inference_input.json'
with open(input_path, 'w') as f:
    json.dump(samples, f, indent=2)
print(f'Input saved to {input_path} ({len(samples)} samples)')

# Restore checkpoint if needed
if not os.path.exists(CKPT_REPO[INFER_MODEL]):
    if not restore_checkpoint_from_drive(INFER_MODEL):
        raise FileNotFoundError(f'No checkpoint for {INFER_MODEL}. Train first.')

# Run inference
!python src/main.py --mode infer --model {INFER_MODEL} --input {input_path}

# Save to Drive
pred_src = 'outputs/predictions/inference_results.json'
if os.path.exists(pred_src):
    shutil.copy2(pred_src, f'{OUTPUT_ROOT}/inference/inference_results_{INFER_MODEL}.json')
    shutil.copy2('outputs/predictions/inference_results.csv',
                 f'{OUTPUT_ROOT}/inference/inference_results_{INFER_MODEL}.csv')
    print(f'\n✓ Predictions saved to Drive')

    # Pretty-print results
    results = json.load(open(pred_src))
    print('\n── Inference Results ──')
    for r in results:
        print(f"\n  Q: {r['question'][:80]}")
        print(f"  → label={r['verification_label']}  conf={r['confidence']:.3f}  flag={r['safety_flag']}  high_risk={r['is_high_risk']}")
        print(f"  reasoning: {r['reasoning']}")

---
## 📊 Section 17 — Visualize Saved Figures
_Display plots generated during evaluation, uncertainty, and safety stages._

In [ ]:
import os, glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display

os.chdir('/content/BioVerify')

FIG_DIR = 'outputs/figures'

# ── Which model's figures to show ─────────────────────────────────
VIZ_MODEL = 'pubmedbert'   # ← change to any model you have trained

figure_groups = {
    'Training Curves': [
        f'{FIG_DIR}/loss_curves_{VIZ_MODEL}.png',
        f'{FIG_DIR}/f1_curves_{VIZ_MODEL}.png',
    ],
    'Prediction Quality': [
        f'{FIG_DIR}/confusion_matrix_{VIZ_MODEL}.png',
        f'{FIG_DIR}/roc_curves_{VIZ_MODEL}.png',
    ],
    'Cross-Model Comparison': [
        f'{FIG_DIR}/per_class_f1_comparison.png',
        f'{FIG_DIR}/contradiction_metrics_comparison.png',
    ],
    'Uncertainty Detection': [
        f'{FIG_DIR}/confidence_distribution_{VIZ_MODEL}.png',
        f'{FIG_DIR}/threshold_tradeoff_curve_{VIZ_MODEL}.png',
    ],
    'Safety Layer': [
        f'{FIG_DIR}/safety_flag_distribution_{VIZ_MODEL}.png',
    ],
}

for group_name, paths in figure_groups.items():
    existing = [p for p in paths if os.path.exists(p)]
    if not existing:
        print(f'⚠️  {group_name}: no figures found yet (run evaluation first)')
        continue

    fig, axes = plt.subplots(1, len(existing), figsize=(7 * len(existing), 5))
    if len(existing) == 1:
        axes = [axes]
    fig.suptitle(group_name, fontsize=13, fontweight='bold')

    for ax, path in zip(axes, existing):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(os.path.basename(path), fontsize=8)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_ROOT}/visualizations/{group_name.replace(" ","_")}.png',
                dpi=120, bbox_inches='tight')
    plt.show()
    print(f'✓ {group_name} saved to Drive')

---
## 💾 Section 18 — Save / Restore Everything (Full Sync)
_Use after completing any set of experiments to persist all work._

In [ ]:
import shutil, os
os.chdir('/content/BioVerify')

print('Saving all checkpoints and outputs to Google Drive...')

# Checkpoints
for model in ['tfidf_lr', 'distilbert', 'biobert', 'pubmedbert']:
    save_checkpoint_to_drive(model)

# Outputs (figures, tables, predictions)
save_outputs_to_drive()

# Results JSON files
for model in ['tfidf_lr', 'distilbert', 'biobert', 'pubmedbert']:
    src = f'experiments/{model}/results.json'
    if os.path.exists(src):
        shutil.copy2(src, f'{RESULTS_ROOT}/{model}_results.json')
        print(f'  ✓ {model} results.json saved')

# Log files
for model in ['tfidf_lr', 'distilbert', 'biobert', 'pubmedbert']:
    for fname in ['train_log.csv', 'training_config.json']:
        src = f'experiments/{model}/logs/{fname}'
        if os.path.exists(src):
            shutil.copy2(src, f'{LOG_ROOT}/{model}_{fname}')

print('\n✓ All data saved to Google Drive.')

In [ ]:
# ── RESTORE from Drive (run at start of new session after re-cloning) ─
import shutil, os
os.chdir('/content/BioVerify')

print('Restoring all checkpoints and outputs from Google Drive...')

for model in ['tfidf_lr', 'distilbert', 'biobert', 'pubmedbert']:
    restore_checkpoint_from_drive(model)

restore_outputs_from_drive()

for model in ['tfidf_lr', 'distilbert', 'biobert', 'pubmedbert']:
    src = f'{RESULTS_ROOT}/{model}_results.json'
    dst = f'experiments/{model}/results.json'
    if os.path.exists(src):
        os.makedirs(f'experiments/{model}', exist_ok=True)
        shutil.copy2(src, dst)
        print(f'  ✓ {model} results.json restored')

print('\n✓ Session restored from Google Drive.')

---
## 📝 Quick Reference

### CLI modes (`src/main.py --mode <mode>`)
| Mode | What it does |
|------|--------------|
| `train` | Train a model; saves checkpoint to `experiments/<model>/checkpoints/` |
| `evaluate` | Evaluate on test set; saves metrics, plots, tables to `outputs/` |
| `uncertainty` | Tune τ on val set; apply to test; compute ECE |
| `safety` | Assign safety flags; compute unsafe catch rate and false safe rate |
| `compare` | Load all results.json → comparison tables + ablation study |
| `infer` | Run full pipeline on a JSON input file |

### Common args
| Arg | Values |
|-----|--------|
| `--model` | `tfidf_lr` \| `distilbert` \| `biobert` \| `pubmedbert` |
| `--epochs` | integer (default 10) |
| `--max_steps` | integer — cap batches/epoch for fast dev runs |
| `--input` | path to `.json` input file (infer mode only) |

### If you run out of memory (OOM)
1. Run the **Memory Cleanup** cell above.
2. Go to **Runtime → Restart runtime**.
3. Re-run Sections 1–6 (mount, GPU, clone, install, data, config, helpers).
4. Restore checkpoints from Drive using **Section 18 restore**.
5. Continue from where you left off.

### Reducing memory for BioBERT / PubMedBERT
Set `BATCH_SIZE = 8` and `MAX_STEPS = None` — gradient accumulation (×2 steps) is already enabled in `src/config.py`, giving effective batch size = 16.